In [ ]:
#https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-PIT-Counts-by-CoC.xlsb
#https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-HIC-Counts-by-CoC.xlsx

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import io
import requests
from google.colab import drive
import warnings



In [2]:
!pip install pyxlsb -q          # -q = quiet, hides the install messages

import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

folder = '/content/drive/MyDrive/'

# sheet_name=None loads every tab into a dictionary: {tab name: table}
pit_sheets = pd.read_excel(folder + '2007-2025-PIT-Counts-by-CoC.xlsb',
                           sheet_name=None, engine='pyxlsb')

print(list(pit_sheets.keys()))

Mounted at /content/drive
['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', 'Table_categories', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', 'CoC Mergers', 'CoCs, States, DDs', 'field list, all years', 'Generate_Table', 'PIT_Counts_Table_Template', 'PIT_Counts_Table', 'Generate_Chart', 'Chart_categories', 'PIT_Counts_Chart', 'PIT_Counts_Chart_Template']


In [3]:
# Tab names that are all digits ('2024') are years. Everything else is notes/tools.
year_tabs = [name for name in pit_sheets if name.isdigit()]
print(year_tabs)

['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007']


In [4]:
pit_2024 = pit_sheets['2024']

print(pit_2024.shape)        # (rows, columns): about 390 regions, about 1,000 columns
pit_2024.iloc[:5, :8]        # first 5 rows, first 8 columns only, so it's readable

(390, 1018)


,CoC Number,CoC Name,Count Types,Overall Homeless,Overall Homeless - Under 18,Overall Homeless - Age 18 to 24,Overall Homeless - Age 25 to 34,Overall Homeless - Age 35 to 44
0,AK-500,Anchorage CoC,Sheltered and Unsheltered Count,1709.0,241.0,169.0,305.0,407.0
1,AK-501,Alaska Balance of State CoC,Sheltered and Unsheltered Count,977.0,202.0,59.0,190.0,217.0
2,AL-500,"Birmingham/Jefferson, St. Clair, Shelby Counti...",Sheltered and Unsheltered Count,911.0,89.0,50.0,151.0,174.0
3,AL-501,Mobile City & County/Baldwin County CoC,Sheltered and Unsheltered Count,675.0,155.0,20.0,96.0,113.0
4,AL-502,Florence/Northwest Alabama CoC,Sheltered and Unsheltered Count,210.0,50.0,13.0,47.0,42.0


In [5]:
cols = ['CoC Number', 'CoC Name', 'Overall Homeless',
        'Sheltered Total Homeless', 'Unsheltered Homeless']

# Only keep names that actually exist in this tab (protects against typos)
cols = [c for c in cols if c in pit_2024.columns]
print(cols)

small = pit_2024[cols]
small.head()

['CoC Number', 'CoC Name', 'Overall Homeless', 'Sheltered Total Homeless', 'Unsheltered Homeless']


,CoC Number,CoC Name,Overall Homeless,Sheltered Total Homeless,Unsheltered Homeless
0,AK-500,Anchorage CoC,1709.0,1363.0,346.0
1,AK-501,Alaska Balance of State CoC,977.0,844.0,133.0
2,AL-500,"Birmingham/Jefferson, St. Clair, Shelby Counti...",911.0,516.0,395.0
3,AL-501,Mobile City & County/Baldwin County CoC,675.0,324.0,351.0
4,AL-502,Florence/Northwest Alabama CoC,210.0,156.0,54.0


In [6]:
small[small['CoC Number'] == 'NM-500']

,CoC Number,CoC Name,Overall Homeless,Sheltered Total Homeless,Unsheltered Homeless
249,NM-500,Albuquerque CoC,2740.0,1509.0,1231.0


In [7]:
small.tail()

,CoC Number,CoC Name,Overall Homeless,Sheltered Total Homeless,Unsheltered Homeless
385,NaN,Total,771480.0,497256.0,274224.0
386,NaN,NaN,NaN,NaN,NaN
387,a MO-604 covers territory in both Missouri and...,NaN,NaN,NaN,NaN
388,* File does not contain the imputed race or ra...,NaN,NaN,NaN,NaN
389,"** Beginning in 2024, all variables for ""Non-H...",NaN,NaN,NaN,NaN


In [8]:
# Keep only real CoC codes: 2 letters, dash, 3 digits (e.g. NM-500)
# na=False treats blank CoC Numbers as "no match"
is_real_coc = small['CoC Number'].astype(str).str.match(r'^[A-Z]{2}-\d{3}$', na=False)

small_clean = small[is_real_coc]
print(small.shape, '->', small_clean.shape)   # should drop about 5 rows
small_clean.tail()

(390, 5) -> (384, 5)


,CoC Number,CoC Name,Overall Homeless,Sheltered Total Homeless,Unsheltered Homeless
380,WV-500,Wheeling/Weirton Area CoC,129.0,87.0,42.0
381,WV-501,"Huntington/Cabell, Wayne Counties CoC",278.0,208.0,70.0
382,WV-503,"Charleston/Kanawha, Putnam, Boone, Clay Counti...",335.0,193.0,142.0
383,WV-508,West Virginia Balance of State CoC,1037.0,503.0,534.0
384,WY-500,Wyoming Statewide CoC,501.0,412.0,89.0


In [9]:
pit_2024['Count Types'].value_counts()

,count
Count Types,
Sheltered and Unsheltered Count,363
Sheltered-Only Count,22


In [10]:
for yr in ['2025', '2024', '2022', '2021']:
    print(yr)
    print(pit_sheets[yr]['Count Types'].value_counts(), '\n')

2025
Count Types
Sheltered and Unsheltered Count    357
Sheltered-Only Count                29
Name: count, dtype: int64 

2024
Count Types
Sheltered and Unsheltered Count    363
Sheltered-Only Count                22
Name: count, dtype: int64 

2022
Count Types
Sheltered and Unsheltered Count    371
Sheltered-Only Count                16
Name: count, dtype: int64 

2021
Count Types
Sheltered-Only Count                       162
Sheltered and full unsheltered count       149
Sheltered and partial unsheltered count     76
Name: count, dtype: int64 



In [11]:
needed = ['CoC Number', 'CoC Name', 'Count Types', 'Overall Homeless',
          'Sheltered Total Homeless', 'Unsheltered Homeless']

for yr in year_tabs:
    missing = [c for c in needed if c not in pit_sheets[yr].columns]
    print(yr, 'missing:', missing)

2025 missing: []
2024 missing: []
2023 missing: []
2022 missing: []
2021 missing: []
2020 missing: []
2019 missing: []
2018 missing: []
2017 missing: []
2016 missing: []
2015 missing: []
2014 missing: []
2013 missing: []
2012 missing: []
2011 missing: []
2010 missing: []
2009 missing: []
2008 missing: ['Count Types']
2007 missing: ['Count Types']


## Data decision: drop "Sheltered-Only Count" rows
Our target is the number of UNSHELTERED people. HUD only requires CoCs to count
people sleeping outside every other year, and in 2021 many skipped it because of COVID.
When a CoC did a sheltered-only count, its unsheltered number is 0 or blank because
nobody went out and counted, NOT because nobody was outside. Keeping those rows would
teach the model that some regions have zero unsheltered people when we don't actually know.

In [12]:
# ---------------------------------------------------------------
# Stack every year of PIT into one long table: one row = one CoC in one year
# ---------------------------------------------------------------

needed = ['CoC Number', 'CoC Name', 'Count Types', 'Overall Homeless',
          'Sheltered Total Homeless', 'Unsheltered Homeless']

pieces = []
for yr in year_tabs:
    df = pit_sheets[yr]

    # 2007 and 2008 have no 'Count Types' column, so keep whatever columns exist
    df = df[[c for c in needed if c in df.columns]].copy()
    df['year'] = int(yr)
    pieces.append(df)

pit_long = pd.concat(pieces, ignore_index=True)
print('Stacked:', pit_long.shape)

# ---------------------------------------------------------------
# Remove the national "Total" row and the footnote rows at the bottom of each tab.
# Real CoC codes start with 2 letters, a dash, 3 digits (e.g. NM-500).
# No '$' at the end, so codes with a footnote letter like 'MO-604a' are kept.
# ---------------------------------------------------------------
is_real_coc = pit_long['CoC Number'].astype(str).str.match(r'^[A-Z]{2}-\d{3}', na=False)
pit_long = pit_long[is_real_coc].copy()
pit_long['CoC Number'] = pit_long['CoC Number'].str[:6]   # 'MO-604a' -> 'MO-604'
print('After removing Total/footnotes:', pit_long.shape)

# ---------------------------------------------------------------
# WHY WE DROP "Sheltered-Only Count" ROWS:
# Our target is UNSHELTERED people. In a sheltered-only year, the CoC did not
# go out and count people sleeping outside. Its unsheltered number is 0 or blank
# because nobody counted, NOT because nobody was there. Keeping these rows would
# give the model fake zeros. HUD only requires the street count every other year,
# and in 2021 many CoCs skipped it for COVID, so this mostly removes even years and 2021.
#
# NOTE: 2007-2008 have no 'Count Types' column (NaN), so they are KEPT.
# NOTE: 2021 also has "Sheltered and partial unsheltered count" rows. These are KEPT for now,
#       but their unsheltered numbers are incomplete. Revisit this.
# ---------------------------------------------------------------
before = len(pit_long)
pit_long = pit_long[pit_long['Count Types'] != 'Sheltered-Only Count'].copy()
print(f'Dropped {before - len(pit_long)} sheltered-only rows ->', pit_long.shape)

Stacked: (7491, 7)
After removing Total/footnotes: (7431, 7)
Dropped 633 sheltered-only rows -> (6798, 7)


In [13]:
# How many regions are left in each year?
pit_long['year'].value_counts().sort_index()

,count
year,
2007,394
2008,390
2009,391
2010,348
2011,392
2012,268
2013,392
2014,305
2015,395


In [14]:
# Albuquerque's remaining years
pit_long[pit_long['CoC Number'] == 'NM-500'][['year', 'Count Types', 'Unsheltered Homeless']]

,year,Count Types,Unsheltered Homeless
249,2025,Sheltered and Unsheltered Count,1367.0
638,2024,Sheltered and Unsheltered Count,1231.0
1028,2023,Sheltered and Unsheltered Count,977.0
1419,2022,Sheltered and Unsheltered Count,163.0
1809,2021,Sheltered and full unsheltered count,413.0
2602,2019,Sheltered and Unsheltered Count,567.0
3400,2017,Sheltered and Unsheltered Count,384.0
4195,2015,Sheltered and Unsheltered Count,183.0
4983,2013,Sheltered and Unsheltered Count,144.0
5772,2011,Sheltered and Unsheltered Count,387.0


In [15]:
# Every Count Types label, across all years (dropna=False also shows the blanks)
pit_long['Count Types'].value_counts(dropna=False)

,count
Count Types,
Sheltered and Unsheltered Count,5678
NaN,784
Sheltered and full unsheltered count,149
No 2010 PIT Count,89
Sheltered and partial unsheltered count,76
Sheltered-Only Count*,22


In [16]:
# For each CoC, line up its 2007 and 2008 unsheltered numbers side by side
u07 = pit_long[pit_long['year'] == 2007].set_index('CoC Number')['Unsheltered Homeless']
u08 = pit_long[pit_long['year'] == 2008].set_index('CoC Number')['Unsheltered Homeless']

both = pd.concat([u07, u08], axis=1, keys=['2007', '2008']).dropna()
same = (both['2007'] == both['2008']).sum()
print(f'{same} of {len(both)} CoCs have the identical number in 2007 and 2008')

131 of 389 CoCs have the identical number in 2007 and 2008


### NOTE: NM-500 2022 unsheltered = 163, far below 2021 (413) and 2023 (977).
### Possibly a change in count method, not a real drop. Kept, but flagged in write-up.